# 04 — Jev as the decision engine (before any logical route)

Jev (TypeSafe System One) is the **only** classifier. After `Choice` returns `ollama` or `openai`, dispatch is dumb — no regex, no LiteLLM pick.

Docs: [introducing System One / Jev](https://typesafe.ai/blog/introducing-system-one-models-and-jev)

This notebook **fails immediately** if `TYPESAFE_API_KEY` is missing. There is no silent fallback to baseline.

Kernel: Python 3.10+ (`typesafe-sdk` requires it).


In [ ]:
%pip install typesafe-sdk python-dotenv -q


In [ ]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

cwd = Path.cwd()
poc_dir = None
root = None
for p in [cwd, *cwd.parents]:
    if (p / "eval_queries.py").exists():
        poc_dir = p
        break
    if (p / "poc" / "eval_queries.py").exists():
        poc_dir = p / "poc"
        break
if poc_dir is None:
    raise FileNotFoundError("eval_queries.py not found — run from route-chatbot/ or route-chatbot/poc/")
sys.path.insert(0, str(poc_dir))

for p in [cwd, *cwd.parents]:
    if (p / ".env").exists() and (p / "main.py").exists():
        root = p
        load_dotenv(p / ".env")
        break
else:
    load_dotenv()

from eval_queries import EVAL_QUERIES

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3")
OLLAMA_MODEL_2 = os.getenv("OLLAMA_MODEL_2", "llama3")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-3.5-turbo")
TYPESAFE_API_KEY = os.getenv("TYPESAFE_API_KEY", "")

print("poc_dir", poc_dir)
print("eval queries", len(EVAL_QUERIES))
print("ollama", OLLAMA_BASE_URL, OLLAMA_MODEL, "| alt", OLLAMA_MODEL_2)
print("openai model", OPENAI_MODEL, "| key set", bool(OPENAI_API_KEY))
print("typesafe key set", bool(TYPESAFE_API_KEY.strip()))
print("typesafe key set", bool(TYPESAFE_API_KEY.strip()))


## Require a key, then Choice


In [ ]:
if not TYPESAFE_API_KEY.strip():
    raise RuntimeError(
        "TYPESAFE_API_KEY is missing. Set it in route-chatbot/.env "
        "(TypeSafe console: https://console.typesafe.ai) and re-run. "
        "This notebook will not fall back to regex/LiteLLM."
    )

from typesafe_sdk import Choice, TypeSafeClient

ROUTE_QUESTION = Choice(
    instructions=(
        "Pick which model should answer the user message. "
        "Choose exactly one option."
    ),
    criteria={
        "ollama": (
            "Greetings, small talk, thanks, casual chat, or generic questions "
            "a small local model can handle."
        ),
        "openai": (
            "Coding, algorithms, debugging, analysis, design, or creative writing "
            "that needs a stronger model."
        ),
    },
)


In [ ]:
def decide_route(message: str):
    """Jev first — no other router runs before this. Returns (label, details)."""
    with TypeSafeClient() as client:
        response = client.system_one(
            state=message,
            questions={"route": ROUTE_QUESTION},
        )
    answer = response.choices["route"]
    choice = answer.choice
    confidence = getattr(answer, "confidence", None)
    probs = getattr(answer, "probabilities", None)
    if choice not in ("ollama", "openai"):
        raise RuntimeError(f"unexpected Jev choice: {choice!r}")
    return choice, {"confidence": confidence, "probabilities": probs}


## Eval (each row is a Jev API call — this *is* the router tax)


In [ ]:
rows = []
for item in EVAL_QUERIES:
    t0 = time.perf_counter()
    err = None
    predicted = None
    extra = None
    try:
        predicted, extra = decide_route(item["message"])
    except Exception as e:
        err = f"{type(e).__name__}: {e}"
    ms = (time.perf_counter() - t0) * 1000
    rows.append({
        "message": item["message"],
        "expected": item["expected"],
        "predicted": predicted,
        "match": predicted == item["expected"],
        "latency_ms": round(ms, 1),
        "error": err,
        "extra": extra,
    })

n = len(rows)
ok = sum(1 for r in rows if r["match"])
errs = sum(1 for r in rows if r["error"])
mean_ms = sum(r["latency_ms"] for r in rows) / n if n else 0
print(f"accuracy {ok}/{n} ({100 * ok / n:.0f}%)  mean latency {mean_ms:.1f} ms  errors {errs}")
print()
for r in rows:
    flag = "OK  " if r["match"] else "MISS"
    extra = f"  {r['extra']}" if r["extra"] else ""
    err = f"  ERR {r['error']}" if r["error"] else ""
    print(f"  [{flag}] {r['latency_ms']:7.1f} ms  exp={r['expected']:7} pred={r['predicted']}  {r['message'][:70]}{extra}{err}")


## Notes (fill during the experiment)

- Hosted classifier, no self-host. Decision happens **before** any model route.
- Look at confidence on misses — calibrated uncertainty is the Jev-specific signal.
- Compare latency here to regex (01) and mf (03): this is the router tax.


In [ ]:
GENERATE = False  # flip to True during the experiment, not the scaffold

def call_ollama(message: str, model: str | None = None) -> str:
    import json
    import urllib.request

    payload = json.dumps({
        "model": model or OLLAMA_MODEL,
        "prompt": message,
        "stream": False,
    }).encode()
    req = urllib.request.Request(
        f"{OLLAMA_BASE_URL.rstrip('/')}/api/generate",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        return json.loads(resp.read().decode()).get("response", "").strip()


def call_openai(message: str) -> str:
    import json
    import urllib.request

    if not OPENAI_API_KEY:
        raise RuntimeError("OPENAI_API_KEY is missing")
    payload = json.dumps({
        "model": OPENAI_MODEL,
        "messages": [{"role": "user", "content": message}],
        "max_tokens": 64,
    }).encode()
    req = urllib.request.Request(
        "https://api.openai.com/v1/chat/completions",
        data=payload,
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {OPENAI_API_KEY}",
        },
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=60) as resp:
        data = json.loads(resp.read().decode())
        return data["choices"][0]["message"]["content"].strip()


if GENERATE:
    for item in EVAL_QUERIES[:3]:
        decision = decide_route(item["message"])
        label = decision[0] if isinstance(decision, tuple) else decision
        fn = call_openai if label == "openai" else call_ollama
        print("---", item["message"], "->", label)
        print(fn(item["message"])[:400])
        print()
else:
    print("GENERATE is False — routing only. Flip it to actually call models.")
